# SurfPerch Batch Feature Extractor — Australian Pilot (180 files)

**Objective:** Process all 180 Australian WAV files through the SurfPerch pipeline.

**Pipeline (per file):**
1. Load WAV, record original sample rate and sample count
2. Resample to 32 kHz (SurfPerch requirement)
3. Split into contiguous, non-overlapping 5-second segments
4. Discard any final incomplete segment
5. Run SurfPerch inference → 1280-D embedding per segment
6. Mean-pool across segments → one 1280-D vector per recording

**Source of truth:** `Tutorial/2-Feature_Extraction.ipynb` (model loading, preprocessing, `model.infer_tf`)

**Outputs:**
- `outputs/surfperch_aus_pilot/surfperch_aus_40.csv` — (180 × 1281) feature matrix with filename
- `outputs/surfperch_aus_pilot/surfperch_aus_manifest.csv` — per-file diagnostics

## 1 · Environment Check

In [ ]:
import sys
print("Python:", sys.executable)
print("Version:", sys.version)

import tensorflow as tf
print("\nTensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))

import numpy as np
import librosa
import pandas as pd
import soundfile as sf
import os
import time

print("\nAll imports OK.")

## 2 · Configuration

In [ ]:
# Paths  (relative to repo root — run notebook from repo root or Tutorial/)
# Detect whether CWD is the repo root or Tutorial/
if os.path.isdir('Tutorial/SurfPerch/savedmodel'):
    REPO_ROOT = '.'
elif os.path.isdir('SurfPerch/savedmodel'):
    REPO_ROOT = '..'
else:
    raise FileNotFoundError("Cannot locate SurfPerch/savedmodel. Run from repo root or Tutorial/.")

MODEL_DIR   = os.path.join(REPO_ROOT, 'Tutorial', 'SurfPerch', 'savedmodel')
AUDIO_DIR   = os.path.join(REPO_ROOT, 'data', 'raw_audio_aus_test')
CSV_PATH    = os.path.join(REPO_ROOT, 'data', 'pretrained_CNN_aus.csv')
OUTPUT_DIR  = os.path.join(REPO_ROOT, 'outputs', 'surfperch_aus_pilot')

# SurfPerch constants (from Tutorial/2-Feature_Extraction.ipynb)
TARGET_SR          = 32_000   # Hz
SEGMENT_DURATION   = 5        # seconds
SEGMENT_LENGTH     = TARGET_SR * SEGMENT_DURATION   # 160 000 samples
EMBEDDING_DIM      = 1280

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Model dir  : {os.path.abspath(MODEL_DIR)}")
print(f"Audio dir  : {os.path.abspath(AUDIO_DIR)}")
print(f"CSV path   : {os.path.abspath(CSV_PATH)}")
print(f"Output dir : {os.path.abspath(OUTPUT_DIR)}")
print(f"Segment len: {SEGMENT_LENGTH:,} samples  ({SEGMENT_DURATION}s × {TARGET_SR:,} Hz)")

## 3 · Select 3 Pilot Files (confirmed in `pretrained_CNN_aus.csv`)

In [ ]:
# The pretrained_CNN_aus.csv has WAV filenames as column headers.
# Read just the header row to get the set of known filenames.
csv_columns = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
csv_filenames = {c for c in csv_columns if c.endswith('.wav')}
print(f"WAV filenames in CSV: {len(csv_filenames)}")

# List WAV files on disk
disk_files = sorted(f for f in os.listdir(AUDIO_DIR) if f.lower().endswith('.wav'))
print(f"WAV files on disk  : {len(disk_files)}")

# Intersection
confirmed = sorted(set(disk_files) & csv_filenames)
print(f"Confirmed overlap  : {len(confirmed)}")

# Pick the first 3 alphabetically
PILOT_FILES = confirmed[:3]
print(f"\nPilot files:")
for f in PILOT_FILES:
    print(f"  {f}")

assert len(PILOT_FILES) == 3, f"Expected 3 pilot files, got {len(PILOT_FILES)}"

## 4 · Load SurfPerch SavedModel

In [ ]:
# Load model exactly as in Tutorial/2-Feature_Extraction.ipynb
model = tf.saved_model.load(MODEL_DIR)

# Verify the infer_tf method exists
assert hasattr(model, 'infer_tf'), "Model has no 'infer_tf' method."

# Quick sanity check with a silent 5-second segment
dummy_input = np.zeros((1, SEGMENT_LENGTH), dtype=np.float32)
dummy_logits, dummy_emb = model.infer_tf(dummy_input)
print(f"Sanity check — dummy logits shape : {dummy_logits.shape}")
print(f"Sanity check — dummy embed  shape : {dummy_emb.shape}")
assert dummy_emb.shape == (1, EMBEDDING_DIM), (
    f"Expected (1, {EMBEDDING_DIM}), got {dummy_emb.shape}"
)
print("\nModel loaded and verified.")

## 5 · Helper Functions

Adapted directly from `Tutorial/2-Feature_Extraction.ipynb` with added diagnostics.

In [ ]:
def resample_and_split_audio(file_path, original_sr, target_sr=TARGET_SR,
                              segment_duration=SEGMENT_DURATION):
    """
    Load, resample to target_sr, and split into non-overlapping segments.
    Discards any final incomplete segment.
    Identical logic to Tutorial/2-Feature_Extraction.ipynb.
    """
    audio, _ = librosa.load(file_path, sr=original_sr)   # Load at native rate
    audio = librosa.resample(audio, orig_sr=original_sr, target_sr=target_sr)

    segment_length = target_sr * segment_duration
    total_segments = len(audio) // segment_length

    segments = []
    for i in range(total_segments):
        start = i * segment_length
        end   = start + segment_length
        segments.append(audio[start:end])

    return audio, segments


def extract_surfperch_embeddings(file_path, model):
    """
    Full per-file pipeline:
      1. Read original metadata via soundfile
      2. Resample + split (librosa, matching tutorial)
      3. Run model.infer_tf on each segment
      4. Stack segment embeddings
      5. Mean-pool to a single 1280-D vector
    Returns a diagnostics dict.
    """
    filename = os.path.basename(file_path)

    # --- 1. Original metadata (without loading full audio into memory twice) ---
    info = sf.info(file_path)
    original_sr     = info.samplerate
    original_frames = info.frames

    # --- 2. Resample and split ---
    resampled_audio, segments = resample_and_split_audio(
        file_path, original_sr=original_sr
    )
    resampled_count = len(resampled_audio)
    n_segments      = len(segments)

    # --- 3. Inference per segment ---
    seg_embeddings = []
    for seg in segments:
        # Model expects (batch, 160000) — add batch dimension
        logits, emb = model.infer_tf(seg[np.newaxis, :])
        seg_embeddings.append(emb.numpy()[0])   # shape (1280,)

    # --- 4. Stack ---
    seg_matrix = np.stack(seg_embeddings, axis=0)   # (N, 1280)

    # --- 5. Mean-pool ---
    pooled = seg_matrix.mean(axis=0)                 # (1280,)

    # --- 6. NaN / Inf check ---
    has_nan_inf = bool(np.isnan(pooled).any() or np.isinf(pooled).any())

    return {
        'filename':             filename,
        'original_sr':          original_sr,
        'original_samples':     original_frames,
        'resampled_samples':    resampled_count,
        'n_segments':           n_segments,
        'seg_embedding_shape':  seg_matrix.shape,
        'pooled_shape':         pooled.shape,
        'has_nan_inf':          has_nan_inf,
        'seg_matrix':           seg_matrix,
        'pooled':               pooled,
    }


print("Helper functions defined.")

## 6 · Process 3 Pilot Files

In [ ]:
results = []
total_segments = 0

for idx, fname in enumerate(PILOT_FILES, 1):
    fpath = os.path.join(AUDIO_DIR, fname)
    print(f"\n{'='*70}")
    print(f"  File {idx}/3: {fname}")
    print(f"{'='*70}")

    r = extract_surfperch_embeddings(fpath, model)
    results.append(r)

    # --- Per-file diagnostics ---
    print(f"  Filename              : {r['filename']}")
    print(f"  Original sample rate  : {r['original_sr']} Hz")
    print(f"  Original sample count : {r['original_samples']:,}")
    print(f"  Resampled sample count: {r['resampled_samples']:,}")
    print(f"  Complete 5s segments  : {r['n_segments']}")
    print(f"  Segment embed shape   : {r['seg_embedding_shape']}")
    print(f"  Pooled embed shape    : {r['pooled_shape']}")
    print(f"  NaN/Inf present       : {r['has_nan_inf']}")

    # --- Per-file assertions ---
    assert r['seg_embedding_shape'] == (r['n_segments'], EMBEDDING_DIM), (
        f"Segment shape mismatch: {r['seg_embedding_shape']}"
    )
    assert r['pooled_shape'] == (EMBEDDING_DIM,), (
        f"Pooled shape mismatch: {r['pooled_shape']}"
    )
    assert not r['has_nan_inf'], f"NaN/Inf detected in pooled embedding for {fname}"

    total_segments += r['n_segments']
    print(f"  ✓ All assertions passed.")

print(f"\n{'='*70}")
print(f"  All 3 pilot files processed successfully.")
print(f"{'='*70}")

## 7 · Pilot Validation Summary

In [ ]:
# Stack all pooled embeddings into the final feature matrix
pooled_matrix = np.stack([r['pooled'] for r in results], axis=0)
result_filenames = [r['filename'] for r in results]

print("=" * 60)
print("  PILOT VALIDATION SUMMARY (3 files)")
print("=" * 60)
print(f"  Files processed          : {len(results)}")
print(f"  Total segments processed : {total_segments}")
print(f"  Pooled feature matrix    : {pooled_matrix.shape}")
print(f"  Filenames preserved      :")
for i, fn in enumerate(result_filenames):
    match = fn == PILOT_FILES[i]
    print(f"    [{i}] {fn}  (exact match: {match})")
print()

# --- Final assertions ---
assert len(results) == 3, f"Expected 3 files, got {len(results)}"
assert pooled_matrix.shape == (3, EMBEDDING_DIM), (
    f"Expected (3, {EMBEDDING_DIM}), got {pooled_matrix.shape}"
)
assert result_filenames == PILOT_FILES, (
    f"Filename mismatch: {result_filenames} != {PILOT_FILES}"
)
assert not np.isnan(pooled_matrix).any(), "NaN in pooled matrix"
assert not np.isinf(pooled_matrix).any(), "Inf in pooled matrix"

print("  ✓ All pilot assertions passed.")
print("  ✓ Proceeding to full 180-file batch.")
print("=" * 60)

In [ ]:
# Quick peek at the pooled feature matrix
pd.DataFrame(
    pooled_matrix,
    index=result_filenames,
    columns=[f'emb_{i}' for i in range(EMBEDDING_DIM)]
).iloc[:, :8].round(4)

---
## 8 · Full Batch: Process All 180 Australian Files

Uses the **identical** extraction methodology validated in the 3-file pilot above.
- Same `resample_and_split_audio` + `extract_surfperch_embeddings` functions
- Same 32 kHz resample, 5 s non-overlapping segments, remainder discarded
- Same `model.infer_tf`, same mean-pooling to 1280-D

In [ ]:
# Full file list — all 180 confirmed files (sorted alphabetically)
ALL_FILES = confirmed  # already sorted, all 180 confirmed against CSV
N_TOTAL = len(ALL_FILES)

assert N_TOTAL == 180, f"Expected 180 files, got {N_TOTAL}"
print(f"Files to process: {N_TOTAL}")
print(f"First: {ALL_FILES[0]}")
print(f"Last : {ALL_FILES[-1]}")

In [ ]:
# --- Process all 180 files with progress reporting ---
all_results = []
all_total_segments = 0
REPORT_EVERY = 10   # print progress every N files

batch_start = time.time()

for idx, fname in enumerate(ALL_FILES, 1):
    fpath = os.path.join(AUDIO_DIR, fname)
    file_start = time.time()

    r = extract_surfperch_embeddings(fpath, model)
    all_results.append(r)

    file_elapsed = time.time() - file_start

    # Per-file assertions (same as pilot)
    assert r['seg_embedding_shape'] == (r['n_segments'], EMBEDDING_DIM), (
        f"Segment shape mismatch for {fname}: {r['seg_embedding_shape']}"
    )
    assert r['pooled_shape'] == (EMBEDDING_DIM,), (
        f"Pooled shape mismatch for {fname}: {r['pooled_shape']}"
    )
    assert not r['has_nan_inf'], f"NaN/Inf detected in {fname}"

    all_total_segments += r['n_segments']

    # Progress reporting
    if idx % REPORT_EVERY == 0 or idx == N_TOTAL or idx == 1:
        elapsed = time.time() - batch_start
        rate = idx / elapsed
        eta = (N_TOTAL - idx) / rate if rate > 0 else 0
        print(
            f"  [{idx:3d}/{N_TOTAL}]  {fname}  "
            f"segs={r['n_segments']:2d}  "
            f"{file_elapsed:.1f}s  "
            f"total={elapsed:.0f}s  "
            f"ETA={eta:.0f}s"
        )

batch_elapsed = time.time() - batch_start
print(f"\n{'='*70}")
print(f"  All {N_TOTAL} files processed in {batch_elapsed:.1f}s")
print(f"  Total segments: {all_total_segments}")
print(f"{'='*70}")

## 9 · Build Feature Matrix and Manifest

In [ ]:
# --- Build the (180, 1280) pooled feature matrix ---
all_pooled = np.stack([r['pooled'] for r in all_results], axis=0)
all_filenames = [r['filename'] for r in all_results]

print(f"Pooled matrix shape: {all_pooled.shape}")
print(f"Filenames count    : {len(all_filenames)}")
print(f"Unique filenames   : {len(set(all_filenames))}")

# --- Build features DataFrame: filename + feature_0 .. feature_1279 ---
feature_cols = [f'feature_{i}' for i in range(EMBEDDING_DIM)]
features_df = pd.DataFrame(all_pooled, columns=feature_cols)
features_df.insert(0, 'filename', all_filenames)

print(f"\nFeatures DataFrame shape: {features_df.shape}")
features_df.head(3)

In [ ]:
# --- Build the manifest DataFrame ---
manifest_rows = []
for r in all_results:
    manifest_rows.append({
        'filename':               r['filename'],
        'original_sample_rate':   r['original_sr'],
        'original_sample_count':  r['original_samples'],
        'resampled_sample_count': r['resampled_samples'],
        'n_complete_5s_segments': r['n_segments'],
        'pooled_embedding_dim':   EMBEDDING_DIM,
    })

manifest_df = pd.DataFrame(manifest_rows)
print(f"Manifest shape: {manifest_df.shape}")
manifest_df.head()

## 10 · Save Outputs

In [ ]:
# --- Save feature CSV ---
features_path = os.path.join(OUTPUT_DIR, 'surfperch_aus_40.csv')
features_df.to_csv(features_path, index=False)
print(f"Features saved: {os.path.abspath(features_path)}")
print(f"  Shape: {features_df.shape}")
print(f"  Size : {os.path.getsize(features_path):,} bytes")

# --- Save manifest CSV ---
manifest_path = os.path.join(OUTPUT_DIR, 'surfperch_aus_manifest.csv')
manifest_df.to_csv(manifest_path, index=False)
print(f"\nManifest saved: {os.path.abspath(manifest_path)}")
print(f"  Shape: {manifest_df.shape}")
print(f"  Size : {os.path.getsize(manifest_path):,} bytes")

## 11 · Reload and Verify Saved CSV

In [ ]:
# Reload from disk to ensure round-trip integrity
reloaded = pd.read_csv(features_path)
print(f"Reloaded shape: {reloaded.shape}")

assert reloaded.shape == (180, 1281), f"Expected (180, 1281), got {reloaded.shape}"
assert list(reloaded.columns[:2]) == ['filename', 'feature_0']
assert list(reloaded.columns[-1:]) == ['feature_1279']
assert reloaded['filename'].tolist() == all_filenames

# Verify numeric values match
reloaded_matrix = reloaded.iloc[:, 1:].values
max_diff = np.abs(reloaded_matrix - all_pooled).max()
print(f"Max round-trip difference: {max_diff:.2e}")
assert max_diff < 1e-5, f"Round-trip error too large: {max_diff}"

print("✓ Saved CSV verified — round-trip intact.")

## 12 · Final Validation Summary

In [ ]:
print("=" * 70)
print("  FINAL VALIDATION SUMMARY — 180-FILE AUSTRALIAN PILOT")
print("=" * 70)

# 1. Exactly 180 files
n_files = len(all_results)
print(f"  Files processed           : {n_files}")
assert n_files == 180, f"Expected 180, got {n_files}"

# 2. Exactly 180 unique filenames
n_unique = len(set(all_filenames))
print(f"  Unique filenames          : {n_unique}")
assert n_unique == 180, f"Expected 180 unique, got {n_unique}"

# 3. Exactly 1280 features per recording
print(f"  Features per recording    : {all_pooled.shape[1]}")
assert all_pooled.shape[1] == EMBEDDING_DIM, (
    f"Expected {EMBEDDING_DIM}, got {all_pooled.shape[1]}"
)

# 4. No NaN/Inf
has_nan = bool(np.isnan(all_pooled).any())
has_inf = bool(np.isinf(all_pooled).any())
print(f"  Any NaN                   : {has_nan}")
print(f"  Any Inf                   : {has_inf}")
assert not has_nan, "NaN detected in feature matrix"
assert not has_inf, "Inf detected in feature matrix"

# 5. All 180 filenames match pretrained_CNN_aus.csv
missing_from_csv = set(all_filenames) - csv_filenames
print(f"  Missing from CSV          : {len(missing_from_csv)}")
assert len(missing_from_csv) == 0, f"Files not in CSV: {missing_from_csv}"

# 6. Feature matrix shape
print(f"  Output feature matrix     : {all_pooled.shape}")
assert all_pooled.shape == (180, EMBEDDING_DIM), (
    f"Expected (180, {EMBEDDING_DIM}), got {all_pooled.shape}"
)

# 7. Total segments
print(f"  Total segments processed  : {all_total_segments}")

# 8. Timing
print(f"  Total processing time     : {batch_elapsed:.1f}s")
print(f"  Average per file          : {batch_elapsed / n_files:.2f}s")

# 9. Output files
print(f"\n  Feature CSV  : {os.path.abspath(features_path)}")
print(f"  Manifest CSV : {os.path.abspath(manifest_path)}")

print(f"\n{'='*70}")
print("  ✓ All 6 final assertions passed.")
print("  ✓ 180-file Australian pilot COMPLETE.")
print("  ✗ Do NOT proceed beyond this 180-file pilot.")
print(f"{'='*70}")

In [ ]:
# Final peek: first 5 rows, first 8 features
features_df.iloc[:5, :9]

In [ ]:
# Final peek: manifest summary statistics
manifest_df.describe()